## In sample Param Grid Search

In [ ]:
import sys
from pathlib import Path
import importlib

# Add parent directory to path so we can import from strategies/
sys.path.insert(0, str(Path.cwd().parent))

import strategies.supertrend_ema_confirmation.strategy as strategy_mod
importlib.reload(strategy_mod)

Strategy = strategy_mod.SupertrendEmaConfirmationStrategy

## Constants

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from investing_algorithm_framework import BacktestDateRange

data_storage_path = Path.cwd().parent / "data"
backtest_results_dir = Path.cwd().parent / "backtest_results"
backtest_results_dir_in_sample = backtest_results_dir / "in_sample"
reports_dir = Path.cwd().parent / "reports"
figures_dir = reports_dir / "figures"

backtest_window_date_range = BacktestDateRange(
    start_date=datetime(2022, 1, 1, tzinfo=timezone.utc),
    end_date=datetime(2025, 12, 30, tzinfo=timezone.utc)
)

MARKET = "BITVAVO"
in_sample_assets = ["BTC", "ETH", "ADA", "SOL", "DOT"]
time_frames = ["2h", "4h", "1d"]

In [ ]:
from investing_algorithm_framework import generate_rolling_backtest_windows

rolling_backtest_windows = generate_rolling_backtest_windows(
    start_date=backtest_window_date_range.start_date,
    end_date=backtest_window_date_range.end_date,
    train_days=365,
    test_days=180,
    gap_days=30,
    step_days=90,
)

In [ ]:
from itertools import product
from investing_algorithm_framework import create_markdown_table


# ══════════════════════════════════════════════════════════════════
#  BASE PARAMS — SuperTrend + EMA Confirmation Strategy
# ══════════════════════════════════════════════════════════════════
params = {
    # ── Timeframes ───────────────────────────────────────────────
    "ema_timeframe": "2h",
    "rsi_timeframe": "2h",

    # ── EMA settings ─────────────────────────────────────────────
    "ema_short_period": 50,
    "ema_long_period": 200,
    "ema_cross_lookback_window": 48,

    # ── RSI settings ─────────────────────────────────────────────
    "rsi_period": 14,
    "rsi_overbought_threshold": 75,
    "rsi_oversold_threshold": 30,

    # ── SuperTrend (primary trend filter) ────────────────────────
    "supertrend_atr_length": 10,
    "supertrend_factor": 3.0,
    "use_supertrend_filter": True,

    # ── Bollinger Bands (overextension guardrail) ────────────────
    "bollinger_period": 20,
    "bollinger_std_dev": 2.0,
    "use_bollinger_filter": True,

    # ── Risk management ──────────────────────────────────────────
    "stop_loss_percentage": 5.0,
    "take_profit_percentage": 10.0,
    "trailing_stop_loss": True,

    # ── Cooldown rules (in bars of ema_timeframe) ───────────────
    "reentry_cooldown_bars": 12,
    "portfolio_cooldown_bars": 2,
}


# ══════════════════════════════════════════════════════════════════
#  GRID PROFILES (target ≤ 100 combinations)
# ══════════════════════════════════════════════════════════════════

# ── Q0: Timeframe ────────────────────────────────────────────────
timeframe_profiles = {
    "2h": {"ema_timeframe": "2h", "rsi_timeframe": "2h"},
    "4h": {"ema_timeframe": "4h", "rsi_timeframe": "4h"},
}

# ── Q1: EMA period pairs ────────────────────────────────────────
ema_period_profiles = {
    "fast_21_100": {
        "ema_short_period": 21,
        "ema_long_period": 100,
        "ema_cross_lookback_window": 24,
    },
    "default_50_200": {
        "ema_short_period": 50,
        "ema_long_period": 200,
        "ema_cross_lookback_window": 48,
    },
}

# ── Q2: SuperTrend ───────────────────────────────────────────────
supertrend_profiles = {
    "responsive_10_2_5": {
        "supertrend_atr_length": 10,
        "supertrend_factor": 2.5,
    },
    "default_10_3_0": {
        "supertrend_atr_length": 10,
        "supertrend_factor": 3.0,
    },
}

# ── Q3: RSI thresholds ──────────────────────────────────────────
rsi_threshold_profiles = {
    "tight_70_30": {
        "rsi_overbought_threshold": 70,
        "rsi_oversold_threshold": 30,
    },
    "wide_75_25": {
        "rsi_overbought_threshold": 75,
        "rsi_oversold_threshold": 25,
    },
}

# ── Q4: Risk management ─────────────────────────────────────────
risk_profiles = {
    "tight_sl3_tp8": {
        "stop_loss_percentage": 3.0,
        "take_profit_percentage": 8.0,
        "trailing_stop_loss": True,
    },
    "default_sl5_tp10": {
        "stop_loss_percentage": 5.0,
        "take_profit_percentage": 10.0,
        "trailing_stop_loss": True,
    },
}

# ── Q5: Cooldown rules ──────────────────────────────────────────
cooldown_profiles = {
    "none": {
        "reentry_cooldown_bars": 0,
        "portfolio_cooldown_bars": 0,
    },
    "light": {
        "reentry_cooldown_bars": 6,
        "portfolio_cooldown_bars": 1,
    },
    "strict": {
        "reentry_cooldown_bars": 12,
        "portfolio_cooldown_bars": 2,
    },
}


# ══════════════════════════════════════════════════════════════════
#  BUILD PARAMETER VARIATIONS (cartesian product of profiles)
# ══════════════════════════════════════════════════════════════════
# Total: 2 × 2 × 2 × 2 × 2 × 3 = 96 combinations

grid_dimensions = {
    "timeframe":       timeframe_profiles,
    "ema_periods":     ema_period_profiles,
    "supertrend":      supertrend_profiles,
    "rsi_thresholds":  rsi_threshold_profiles,
    "risk":            risk_profiles,
    "cooldowns":       cooldown_profiles,
}

dim_names = list(grid_dimensions.keys())
dim_items = [list(grid_dimensions[d].items()) for d in dim_names]

param_variations = []
for combo in product(*dim_items):
    variant = dict(params)  # start from base params
    profile_tag = {}
    for dim_name, (profile_name, overrides) in zip(dim_names, combo):
        variant.update(overrides)
        profile_tag[dim_name] = profile_name
    # Store profile tags for post-hoc analysis
    variant["_grid_profile"] = profile_tag
    param_variations.append(variant)

assert len(param_variations) <= 100, (
    f"Grid has {len(param_variations)} combinations — exceeds 100 cap"
)


# ══════════════════════════════════════════════════════════════════
#  PRINT OVERVIEW
# ══════════════════════════════════════════════════════════════════

print(f"Total parameter combinations: {len(param_variations)}")

# ── Default params ───────────────────────────────────────────────
param_groups = {
    "Timeframes":       ["ema_timeframe", "rsi_timeframe"],
    "EMA":              ["ema_short_period", "ema_long_period", "ema_cross_lookback_window"],
    "RSI":              ["rsi_period", "rsi_overbought_threshold", "rsi_oversold_threshold"],
    "SuperTrend":       ["supertrend_atr_length", "supertrend_factor", "use_supertrend_filter"],
    "Bollinger":        ["bollinger_period", "bollinger_std_dev", "use_bollinger_filter"],
    "Risk management":  ["stop_loss_percentage", "take_profit_percentage", "trailing_stop_loss"],
    "Cooldowns":        ["reentry_cooldown_bars", "portfolio_cooldown_bars"],
}

print(f"\n{'═'*70}")
print(f"  DEFAULT BASE PARAMS (before grid overrides)")
print(f"{'═'*70}")

for group_name, keys in param_groups.items():
    matching = {k: params[k] for k in keys if k in params}
    if not matching:
        continue
    print(f"\n  {group_name}:")
    for k, v in matching.items():
        print(f"    {k:35s} = {v}")

# ── Grid dimensions summary table ────────────────────────────────
print(f"\n{'═'*70}")
print(f"  GRID DIMENSIONS ({len(grid_dimensions)} dimensions)")
print(f"{'═'*70}\n")

grid_table = [
    {"Dimension": dim, "Profiles": ", ".join(grid_dimensions[dim].keys()), "Count": len(grid_dimensions[dim])}
    for dim in dim_names
]
print(create_markdown_table(grid_table))

for dim_name, profiles in grid_dimensions.items():
    print(f"\n{'─'*60}")
    print(f"  {dim_name} ({len(profiles)} profiles)")
    print(f"{'─'*60}")
    for profile_name, overrides in profiles.items():
        if overrides:
            param_str = ", ".join(f"{k}={v}" for k, v in overrides.items())
        else:
            param_str = "(no overrides — uses class defaults)"
        print(f"  {profile_name:20s} │ {param_str}")

## Strategy and Backtest Initialization

In [ ]:
from investing_algorithm_framework import TimeUnit, generate_algorithm_id
from investing_algorithm_framework.domain import tqdm

# Map timeframe strings to (time_unit, interval) scheduling pairs.
TIMEFRAME_TO_SCHEDULE = {
    "2h":  (TimeUnit.HOUR, 2),
    "4h":  (TimeUnit.HOUR, 4),
    "1d":  (TimeUnit.DAY, 1),
}


def initialize_strategies(
    strategy_class,
    param_variations,
    symbols,
    market,
    trading_symbol="EUR",
    filter_fn=None
):
    """Initialize multiple SupertrendEmaConfirmationStrategy instances
    with different parameters from the grid search."""
    strategies = []

    for variant in tqdm(
        param_variations, desc="Initializing strategies", colour="green"
    ):
        # Separate strategy params from metadata keys (prefixed with _)
        strategy_params = {k: v for k, v in variant.items() if not k.startswith("_")}

        # Derive scheduling interval from the ema_timeframe
        tf_str = variant.get("ema_timeframe", "2h")
        time_unit, interval = TIMEFRAME_TO_SCHEDULE[tf_str]

        strategy = strategy_class(
            algorithm_id=generate_algorithm_id(params=variant),
            symbols=symbols,
            trading_symbol=trading_symbol,
            time_unit=time_unit,
            market=market,
            interval=interval,
            metadata={
                "params": variant,
                "time_unit": TimeUnit(time_unit).name,
                "interval": interval,
                "symbols": symbols,
                "market": market,
            },
            **strategy_params,
        )
        strategies.append(strategy)

    if filter_fn:
        strategies = [s for s in strategies if filter_fn(s)]

    return strategies

In [ ]:
import os

from investing_algorithm_framework import (
    create_app, RESOURCE_DIRECTORY, PortfolioConfiguration,
    BacktestEvaluationFocus, rank_results,
)
from investing_algorithm_framework.domain import DATA_DIRECTORY

# Configuration constants for progressive pruning
MIN_TRADE_WINDOW_RATIO = 0.7
MAX_UNPROFITABLE_WINDOW_RATIO = 0.5


def window_filter_function(backtests, backtest_date_range):
    """
    Progressive pruning using summary metrics from combine_backtests.
    """
    print(f"\nFiltering {len(backtests)} backtests for window: {backtest_date_range.name}")

    def progressive_filter(backtest_metrics, backtest=None):
        if backtest_metrics.number_of_trades_closed == 0:
            return False

        if backtest is None:
            return True

        summary = backtest.backtest_summary

        if summary is None:
            print(f"Warning: No summary available for backtest {backtest.backtest_id}. Skipping detailed filters.")

        if summary.number_of_windows is None or summary.number_of_windows < 7:
            return True

        trade_window_ratio = summary.number_of_windows_with_trades / summary.number_of_windows

        if trade_window_ratio < MIN_TRADE_WINDOW_RATIO:
            return False

        unprofitable_ratio = (summary.number_of_windows - summary.number_of_profitable_windows) / summary.number_of_windows

        if unprofitable_ratio > MAX_UNPROFITABLE_WINDOW_RATIO:
            return False

        return True

    ranked_backtests = rank_results(
        backtests.copy(),
        filter_fn=progressive_filter,
        focus=BacktestEvaluationFocus.BALANCED,
        backtest_date_range=backtest_date_range,
    )

    print(f"Backtests after filtering: {len(ranked_backtests)}")
    return ranked_backtests


def final_filter_function(backtests):
    """
    Final selection: rank and keep top strategies.
    """
    ranked = rank_results(
        backtests.copy(),
        focus=BacktestEvaluationFocus.BALANCED,
    )
    # Keep top 75 diverse strategies
    return ranked[:75]


# Run backtests
backtest_windows = [window["train_range"] for window in rolling_backtest_windows]

strategies = initialize_strategies(
    strategy_class=Strategy,
    param_variations=param_variations,
    symbols=in_sample_assets,
    market=MARKET,
)

app = create_app(config={RESOURCE_DIRECTORY: "./resources", DATA_DIRECTORY: data_storage_path})
app.add_portfolio_configuration(
    PortfolioConfiguration(initial_balance=1000, market="BITVAVO", trading_symbol="EUR")
)

backtests = app.run_vector_backtests(
    backtest_date_ranges=backtest_windows,
    initial_amount=1000,
    strategies=strategies,
    risk_free_rate=0.027,
    market=MARKET,
    trading_symbol="EUR",
    continue_on_error=False,
    window_filter_function=window_filter_function,
    final_filter_function=final_filter_function,
    use_checkpoints=True,
    backtest_storage_directory=backtest_results_dir,
    show_progress=True,
    n_workers=os.cpu_count() - 3,
    dynamic_position_sizing=True,
)

print(f"\nComplete — {len(backtests)} backtests after filtering")

## Quick filter via SQLite index & open report

In [ ]:
from investing_algorithm_framework import BacktestReport
from investing_algorithm_framework.cli.index_command import (
    build_index, rank_index, format_table,
)
from investing_algorithm_framework.services.backtest_store import (
    LocalDirStore,
)

# 1. Build (or refresh) the Tier-1 SQLite index — no Parquet decoded.
index_path = build_index(
    str(backtest_results_dir),
    show_progress=True,
    incremental=True,
)
print(f"Index written to: {index_path}")

# 2. Rank by Sharpe and keep the top 20 — pure SQL, instant.
top = rank_index(
    str(backtest_results_dir),
    by="sharpe_ratio",
    where="summary_number_of_trades_closed > 5",
    limit=20,
)
print(f"\nTop {len(top)} strategies by Sharpe ratio:\n")
print(format_table(top))

# 3. Materialise just those bundles through the BacktestStore protocol.
store = LocalDirStore(str(backtest_results_dir))
backtests = [store.open(row["bundle_path"]) for row in top]
print(f"\nLoaded {len(backtests)} backtests for detailed analysis")

In [ ]:
from investing_algorithm_framework import BacktestReport

report = BacktestReport(backtests=backtests)
report.save("/tmp/backtest_report.html")
report.show(browser=True)